# Z4 115M batch / LR sweep — Kaggle 10-minute arms

The next-move architecture arms were neutral or negative.  The reference run
sees only 1.23M tokens in 600 seconds, so this round spends the budget on
tokens and optimization:

1. `batch8` — `kaggle10m`, batch 8, fallback 6 then 4 on T4 OOM.
2. `batch6` — `kaggle10m`, batch 6, fallback 4.
3. `lr3e4` — `kaggle10m`, batch 4, LR 3e-4.

Same WikiText-2 bins, same 600-second cap, same eval.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator (T4)."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), "GiB")


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/nanlm_experimental")
if not (REPO / "z3" / "train_v3.py").is_file():
    subprocess.run(["git","clone","--depth","1","https://github.com/shemshallah/nanlm_experimental.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
os.chdir(REPO)
print(REPO)


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","tokenizers","numpy","pyarrow"],check=True)


In [ ]:
import urllib.request
Path("data").mkdir(exist_ok=True)
parts=[]
for split in ("train","valid"):
    url=f"https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/{split}.txt"
    dest=Path(f"data/wikitext_{split}.txt")
    urllib.request.urlretrieve(url,dest)
    parts.append(dest.read_text(encoding="utf-8"))
corpus=Path("data/wikitext_corpus.txt")
corpus.write_text("\n\n".join(parts),encoding="utf-8")
print(corpus, corpus.stat().st_size)
subprocess.run([
    sys.executable,"z3/build_domain_bins.py",
    "--corpus","wikitext=data/wikitext_corpus.txt",
    "--out","data/wikitext_bins",
    "--max-bytes-per-domain","12000000",
    "--train-fraction","0.98","--valid-fraction","0.01",
],check=True)


In [ ]:
import json, time
ARMS=[
    {"name":"batch8","batch":[8,6,4],"extra":[]},
    {"name":"batch6","batch":[6,4],"extra":[]},
    {"name":"lr3e4","batch":[4],"extra":["--lr","3e-4"]},
]
SUMMARY=[]
for arm in ARMS:
    out=Path(f"/kaggle/working/runs/z3_115m_{arm['name']}")
    base=[sys.executable,"z3/train_v3.py","--preset","kaggle10m",
          "--corpus","data/wikitext_bins/train.bin",
          "--val-corpus","data/wikitext_bins/valid.bin",
          "--data-manifest","data/wikitext_bins/manifest.json",
          "--out",str(out),*arm["extra"]]
    print("=== ARM",arm["name"],flush=True)
    t0=time.time()
    try:
        trained=False
        for batch in arm["batch"]:
            cmd=base+["--batch-size",str(batch)]
            print("try batch",batch,flush=True)
            try:
                subprocess.run(cmd,check=True)
                trained=True
                break
            except subprocess.CalledProcessError as exc:
                print("batch",batch,"failed",exc.returncode,flush=True)
        if not trained:
            raise RuntimeError("all batch sizes failed")
        subprocess.run([
            sys.executable,"z3/eval_domains.py",
            "--checkpoint",str(out/"best.pt"),
            "--test-dir","data/wikitext_bins/test",
            "--data-manifest","data/wikitext_bins/manifest.json",
            "--tokenizer","layered","--document-framing","--sidecar","--require-sidecar",
            "--ctx","512","--batch-size","4","--device","cuda",
            "--out",str(out/"test_domains.json"),
        ],check=True)
        result=json.loads((out/"result.json").read_text())
        test=json.loads((out/"test_domains.json").read_text())
        cfg=json.loads((out/"config.json").read_text())
        best=(result.get("best") or {}).get("metrics") or {}
        agg=test.get("aggregate") or {}
        feedback=(result.get("final") or {}).get("feedback") or agg.get("feedback") or {}
        obj=(agg.get("object_memory") or {}).get("object_gate")
        SUMMARY.append({
            "arm":arm["name"],"ok":True,"batch":cfg.get("batch_size"),
            "lr":cfg.get("lr"),"params":result.get("params"),
            "steps":result.get("steps_completed"),"tokens_seen":result.get("tokens_seen"),
            "wall_seconds":result.get("wall_seconds"),"nonfinite":result.get("nonfinite_skips"),
            "best_bpb":best.get("bits_per_byte"),"test_bpb":test.get("macro_bits_per_byte"),
            "object_gate":obj,
            "feedback":{k:v for k,v in feedback.items() if k.endswith("gate") or k=="progress"},
            "elapsed":round(time.time()-t0,1),
        })
    except Exception as exc:
        SUMMARY.append({"arm":arm["name"],"ok":False,"error":repr(exc),"elapsed":round(time.time()-t0,1)})
    print(json.dumps(SUMMARY[-1],sort_keys=True),flush=True)

Path("/kaggle/working/batch_lr_summary.json").write_text(json.dumps(SUMMARY,indent=2,sort_keys=True)+"\n")
print(json.dumps(SUMMARY,indent=2,sort_keys=True))


The final JSON is `/kaggle/working/batch_lr_summary.json`; run artifacts are
under `/kaggle/working/runs/z3_115m_<arm>/`.
